# "THE PRICE IS RIGHT" Capstone Project (Modified)
This notebook recreates the fine-tuning process for day 5 of week 6. The challenge is to experiment with the hyper-parameters e.g. the number of epochs, batchsizes, e.t.c to understand the process and try to beat the best error score of 75.91 to a smaller value. 

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from items import Item
# from pricer.evaluator import evaluate

In [ ]:
# environment

LITE_MODE = True

load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
#Let's avoid curating all our data again! Load in the curated data available on Hugging face:

print(f"Are we on lite mode? i.e The smaller dataset: {LITE_MODE}")

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

len(train), len(test)

In [ ]:
# Save curated data above to CSV for future use to avoid downloading the data from Hugging Face again.
import csv
from pathlib import Path

def save_to_csv(data, file_path):
    # Open the file in write mode ('w') with newline=''
    with open(file_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        # Write all rows at once
        writer.writerows(data)

    if Path(file_path).is_file():
        print(f"CSV file '{file_path}' created successfully using csv.writer().")

save_to_csv(train, "training_data.csv")
save_to_csv(val, "validation_data.csv")
save_to_csv(test, "test_data.csv")


In [ ]:
# Fine-tuning and file uploads require OpenAI's API (OpenRouter does not support /fine_tuning/jobs).
# Use your OpenAI key from https://platform.openai.com/api-keys
openai = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)

In [ ]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)
import random

TRAIN_SIZE = 400
VAL_SIZE = 100
RANDOM_SEED = 42

rng = random.Random(RANDOM_SEED)
shuffled = train[:]
rng.shuffle(shuffled)
fine_tune_train = shuffled[:TRAIN_SIZE]
fine_tune_validation = shuffled[TRAIN_SIZE:TRAIN_SIZE+VAL_SIZE]

len(fine_tune_train), len(fine_tune_validation)

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

messages_for(fine_tune_train[0])

In [ ]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...

def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

print(make_jsonl(train[:3]))

In [ ]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)


write_jsonl(fine_tune_train, "fine_tune_train.jsonl")
write_jsonl(fine_tune_train, "fine_tune_validation.jsonl")


In [ ]:
# Upload the files to OpenAI i.e. https://platform.openai.com/storage/files/

with open("fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

train_file


with open("fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

validation_file

## Step 2 - Fine Tuning

Here, we will adjust the hyperparameters accordingly to get the best results.

In [ ]:
NUMBER_OF_EPOCHS = 2 # Number of epochs to train the model
TRANING_BATCH_SIZE = 2 # Batch size for training

openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": NUMBER_OF_EPOCHS, "batch_size": TRANING_BATCH_SIZE},
    suffix="pricer"
)

In [ ]:
openai.fine_tuning.jobs.list(limit=1)
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

job_id

In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

## Step 3

Test our fine tuned model

In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

fine_tuned_model_name

In [ ]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price only, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

test_messages_for(test[0])

In [ ]:
# The inference function

def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))